# Ensemble Models for Music Virality Prediction
This notebook builds multiple ensemble models (Voting & Stacking) to predict viral tracks based on Spotify audio features and YouTube engagement metrics.

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
from pathlib import Path

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries imported successfully!")

All libraries imported successfully!


## 2. Data Loading

In [2]:
# Load dataset
DATA_PATH = "../data/processed/spotify_tracks_cleaned.csv"
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nViral class distribution:\n{df['viral'].value_counts()}")

Dataset shape: (113999, 128)

First few rows:
   duration_ms  danceability  energy  key  loudness  mode  speechiness  \
0       230666         0.676  0.4610    1    -6.746     0       0.1430   
1       149610         0.420  0.1660    1   -17.235     1       0.0763   
2       210826         0.438  0.3590    0    -9.734     1       0.0557   
3       201933         0.266  0.0596    0   -18.515     1       0.0363   
4       198853         0.618  0.4430    2    -9.681     1       0.0526   

   acousticness  instrumentalness  liveness  ...  track_genre_spanish  \
0        0.0322          0.000001    0.3580  ...                False   
1        0.9240          0.000006    0.1010  ...                False   
2        0.2100          0.000000    0.1170  ...                False   
3        0.9050          0.000071    0.1320  ...                False   
4        0.4690          0.000000    0.0829  ...                False   

   track_genre_study  track_genre_swedish  track_genre_synth-pop  \
0 

## 3. Feature Engineering

### 3.1 Prepare Features and Target

In [10]:
# Separate features and target
X = df.drop(columns=['viral'])
y = df['viral']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Features columns:\n{X.columns.tolist()}")

Features shape: (113999, 127)
Target shape: (113999,)
Features columns:
['duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'explicit_True', 'track_genre_afrobeat', 'track_genre_alt-rock', 'track_genre_alternative', 'track_genre_ambient', 'track_genre_anime', 'track_genre_black-metal', 'track_genre_bluegrass', 'track_genre_blues', 'track_genre_brazil', 'track_genre_breakbeat', 'track_genre_british', 'track_genre_cantopop', 'track_genre_chicago-house', 'track_genre_children', 'track_genre_chill', 'track_genre_classical', 'track_genre_club', 'track_genre_comedy', 'track_genre_country', 'track_genre_dance', 'track_genre_dancehall', 'track_genre_death-metal', 'track_genre_deep-house', 'track_genre_detroit-techno', 'track_genre_disco', 'track_genre_disney', 'track_genre_drum-and-bass', 'track_genre_dub', 'track_genre_dubstep', 'track_genre_edm', 'track_genre_electro', 'track

### 3.2 Label Encode Categorical Features

In [6]:
# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {numeric_cols}")

# Label encode categorical features
label_encoders = {}
X_encoded = X.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    label_encoders[col] = le
    print(f"Encoded {col}: {len(le.classes_)} unique values")

X = X_encoded
print(f"\nFeatures after encoding:\n{X.head()}")

Categorical columns: []
Numeric columns: ['duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

Features after encoding:
   duration_ms  danceability  energy  key  loudness  mode  speechiness  \
0       230666         0.676  0.4610    1    -6.746     0       0.1430   
1       149610         0.420  0.1660    1   -17.235     1       0.0763   
2       210826         0.438  0.3590    0    -9.734     1       0.0557   
3       201933         0.266  0.0596    0   -18.515     1       0.0363   
4       198853         0.618  0.4430    2    -9.681     1       0.0526   

   acousticness  instrumentalness  liveness  ...  track_genre_spanish  \
0        0.0322          0.000001    0.3580  ...                False   
1        0.9240          0.000006    0.1010  ...                False   
2        0.2100          0.000000    0.1170  ...                False   
3        0.9050          0

### 3.3 Create Interaction Features

In [7]:
# Add interaction features
if 'energy' in X.columns and 'danceability' in X.columns:
    X['energy_x_danceability'] = X['energy'] * X['danceability']
    print("Added: energy_x_danceability")

if 'valence' in X.columns and 'tempo' in X.columns:
    X['valence_x_tempo'] = X['valence'] * X['tempo']
    print("Added: valence_x_tempo")

print(f"\nFeatures shape after interactions: {X.shape}")
print(f"New columns: {X.columns.tolist()[-5:]}")

Added: energy_x_danceability
Added: valence_x_tempo

Features shape after interactions: (113999, 129)
New columns: ['track_genre_trip-hop', 'track_genre_turkish', 'track_genre_world-music', 'energy_x_danceability', 'valence_x_tempo']


### 3.4 Train-Test Split

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"\nTraining set viral distribution:\n{y_train.value_counts()}")
print(f"\nTest set viral distribution:\n{y_test.value_counts()}")

Training set size: (91199, 129)
Test set size: (22800, 129)

Training set viral distribution:
viral
0    68983
1    22216
Name: count, dtype: int64

Test set viral distribution:
viral
0    17246
1     5554
Name: count, dtype: int64


### 3.5 Scale Continuous Features

In [9]:
# Scale all features with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

print(f"Scaled training set shape: {X_train_scaled.shape}")
print(f"Scaled test set shape: {X_test_scaled.shape}")
print(f"\nScaled training set stats:\n{X_train_scaled.describe()}")

Scaled training set shape: (91199, 129)
Scaled test set shape: (22800, 129)

Scaled training set stats:
        duration_ms  danceability        energy           key      loudness  \
count  9.119900e+04  9.119900e+04  9.119900e+04  9.119900e+04  9.119900e+04   
mean  -3.817651e-17 -1.321375e-16  2.310847e-16  1.005055e-16  9.333767e-17   
std    1.000005e+00  1.000005e+00  1.000005e+00  1.000005e+00  1.000005e+00   
min   -2.061050e+00 -3.268326e+00 -2.552199e+00 -1.496208e+00 -7.619760e+00   
25%   -5.051368e-01 -6.432609e-01 -6.702856e-01 -9.337289e-01 -3.489809e-01   
50%   -1.417585e-01  7.214132e-02  1.692496e-01 -9.001001e-02  2.515429e-01   
75%    3.155211e-01  7.356192e-01  8.456523e-01  7.537089e-01  6.470558e-01   
max    4.704215e+01  2.402968e+00  1.426563e+00  1.597428e+00  2.540947e+00   

               mode   speechiness  acousticness  instrumentalness  \
count  9.119900e+04  9.119900e+04  9.119900e+04      9.119900e+04   
mean   4.487688e-17  3.428095e-17 -7.354821e-1

## 4. Build Base Learners with Cross-Validation

### 4.1 XGBoost Classifier (Tuned Parameters)

In [11]:
# XGBClassifier with tuned parameters
xgb_model = XGBClassifier(
    n_estimators=1000,
    random_state=RANDOM_STATE,
    verbosity=0,
    eval_metric='logloss'
)

# Cross-validation
cv_split = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
xgb_cv_scores = cross_val_score(xgb_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"XGBoost CV F1-Macro Scores: {xgb_cv_scores}")
print(f"XGBoost Mean CV F1-Macro: {xgb_cv_scores.mean():.4f} (+/- {xgb_cv_scores.std():.4f})")

xgb_model.fit(X_train_scaled, y_train)
print("XGBoost model trained!")

XGBoost CV F1-Macro Scores: [0.77826904 0.76940748 0.77499094 0.77507912 0.76844339]
XGBoost Mean CV F1-Macro: 0.7732 (+/- 0.0037)
XGBoost model trained!


### 4.2 LightGBM Classifier (Tuned)

In [12]:
# LGBMClassifier with tuning
lgb_model = LGBMClassifier(
    n_estimators=500,
    max_depth=15,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    verbose=-1
)

# Cross-validation
lgb_cv_scores = cross_val_score(lgb_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"LightGBM CV F1-Macro Scores: {lgb_cv_scores}")
print(f"LightGBM Mean CV F1-Macro: {lgb_cv_scores.mean():.4f} (+/- {lgb_cv_scores.std():.4f})")

lgb_model.fit(X_train_scaled, y_train)
print("LightGBM model trained!")

LightGBM CV F1-Macro Scores: [0.68995004 0.67959833 0.69433554 0.6864222  0.68249848]
LightGBM Mean CV F1-Macro: 0.6866 (+/- 0.0052)
LightGBM model trained!


### 4.3 CatBoost Classifier (with Categorical Features)

In [13]:
# Identify categorical feature indices (those that were label encoded)
cat_feature_indices = [X_train_scaled.columns.tolist().index(col) for col in categorical_cols if col in X_train_scaled.columns]

print(f"Categorical feature indices: {cat_feature_indices}")

# CatBoostClassifier
catboost_model = CatBoostClassifier(
    iterations=500,
    cat_features=cat_feature_indices,
    random_state=RANDOM_STATE,
    verbose=0
)

# Cross-validation
catboost_cv_scores = cross_val_score(catboost_model, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')

print(f"CatBoost CV F1-Macro Scores: {catboost_cv_scores}")
print(f"CatBoost Mean CV F1-Macro: {catboost_cv_scores.mean():.4f} (+/- {catboost_cv_scores.std():.4f})")

catboost_model.fit(X_train_scaled, y_train)
print("CatBoost model trained!")

Categorical feature indices: []


RuntimeError: Cannot clone object CatBoostClassifier(cat_features=[], iterations=500, random_state=42, verbose=0), as the constructor either does not set or modifies parameter cat_features

### 4.4 Logistic Regression with Polynomial Features (Audio Features Only)

In [14]:
# Select audio features (Spotify audio features)
audio_features = [
    'danceability', 'energy', 'valence', 'acousticness',
    'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo'
]

# Ensure all audio features are in scaled data
available_audio_features = [col for col in audio_features if col in X_train_scaled.columns]
print(f"Available audio features: {available_audio_features}")

# Create polynomial features (degree 2) on audio features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_audio = X_train_scaled[available_audio_features].values
X_test_audio = X_test_scaled[available_audio_features].values

X_train_audio_poly = poly.fit_transform(X_train_audio)
X_test_audio_poly = poly.transform(X_test_audio)

print(f"Polynomial features shape: {X_train_audio_poly.shape}")

# LogisticRegression
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Cross-validation
lr_cv_scores = cross_val_score(lr_model, X_train_audio_poly, y_train, cv=cv_split, scoring='f1_macro')

print(f"LogisticRegression CV F1-Macro Scores: {lr_cv_scores}")
print(f"LogisticRegression Mean CV F1-Macro: {lr_cv_scores.mean():.4f} (+/- {lr_cv_scores.std():.4f})")

lr_model.fit(X_train_audio_poly, y_train)
print("LogisticRegression model trained!")

Available audio features: ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo']
Polynomial features shape: (91199, 54)
LogisticRegression CV F1-Macro Scores: [0.4320716  0.43270932 0.43453092 0.43190679 0.43368762]
LogisticRegression Mean CV F1-Macro: 0.4330 (+/- 0.0010)
LogisticRegression model trained!


## 5. Build Ensemble Models

### 5.1 Soft Voting Classifier

In [ ]:
# Create a voting classifier with soft voting
# Note: LogisticRegression requires separate handling due to polynomial features
voting_clf = VotingClassifier(
    estimators=[
        ('xgb', XGBClassifier(n_estimators=1000, random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
        ('lgb', LGBMClassifier(n_estimators=500, max_depth=15, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1)),
        ('catboost', CatBoostClassifier(iterations=500, cat_features=cat_feature_indices, random_state=RANDOM_STATE, verbose=0))
    ],
    voting='soft',
    n_jobs=-1
)

# Fit on full training data
voting_clf.fit(X_train_scaled, y_train)
print("Soft Voting Classifier trained!")

# Cross-validation on Voting Classifier
voting_cv_scores = cross_val_score(voting_clf, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')
print(f"Voting Classifier CV F1-Macro: {voting_cv_scores.mean():.4f} (+/- {voting_cv_scores.std():.4f})")

### 5.2 Stacking Classifier

In [ ]:
# Create base learners for stacking
base_learners = [
    ('xgb', XGBClassifier(n_estimators=1000, random_state=RANDOM_STATE, verbosity=0, eval_metric='logloss')),
    ('lgb', LGBMClassifier(n_estimators=500, max_depth=15, learning_rate=0.05, random_state=RANDOM_STATE, verbose=-1)),
    ('catboost', CatBoostClassifier(iterations=500, cat_features=cat_feature_indices, random_state=RANDOM_STATE, verbose=0))
]

# Meta-learner
meta_learner = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)

# Stacking classifier with passthrough=True (meta-learner sees original features too)
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    passthrough=True,
    cv=cv_split,
    n_jobs=-1
)

# Fit stacking classifier
stacking_clf.fit(X_train_scaled, y_train)
print("Stacking Classifier trained!")

# Cross-validation on Stacking Classifier
stacking_cv_scores = cross_val_score(stacking_clf, X_train_scaled, y_train, cv=cv_split, scoring='f1_macro')
print(f"Stacking Classifier CV F1-Macro: {stacking_cv_scores.mean():.4f} (+/- {stacking_cv_scores.std():.4f})")

## 6. Model Evaluation

### 6.1 Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, use_poly=False):
    """
    Evaluate a model and return metrics.
    
    Parameters:
    - model: trained model
    - X_test: test features
    - y_test: test target
    - model_name: name of the model for display
    - use_poly: if True, apply polynomial transformation (for LR)
    """
    if use_poly:
        X_test_transformed = poly.transform(X_test)
    else:
        X_test_transformed = X_test
    
    y_pred = model.predict(X_test_transformed)
    y_pred_proba = model.predict_proba(X_test_transformed)[:, 1]
    
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1-Macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, y_pred_proba)
    }
    
    return metrics, y_pred, y_pred_proba

print("Evaluation function defined!")

### 6.2 Evaluate All Models

In [ ]:
# Evaluate all models
results = []
predictions = {}

# Evaluate base learners
metrics_xgb, pred_xgb, pred_proba_xgb = evaluate_model(xgb_model, X_test_scaled, y_test, 'XGBoost')
results.append(metrics_xgb)
predictions['XGBoost'] = (pred_xgb, pred_proba_xgb)

metrics_lgb, pred_lgb, pred_proba_lgb = evaluate_model(lgb_model, X_test_scaled, y_test, 'LightGBM')
results.append(metrics_lgb)
predictions['LightGBM'] = (pred_lgb, pred_proba_lgb)

metrics_catboost, pred_catboost, pred_proba_catboost = evaluate_model(catboost_model, X_test_scaled, y_test, 'CatBoost')
results.append(metrics_catboost)
predictions['CatBoost'] = (pred_catboost, pred_proba_catboost)

metrics_lr, pred_lr, pred_proba_lr = evaluate_model(lr_model, X_test_audio, y_test, 'Logistic Regression', use_poly=True)
results.append(metrics_lr)
predictions['LogisticRegression'] = (pred_lr, pred_proba_lr)

# Evaluate ensemble models
metrics_voting, pred_voting, pred_proba_voting = evaluate_model(voting_clf, X_test_scaled, y_test, 'Voting Classifier')
results.append(metrics_voting)
predictions['Voting'] = (pred_voting, pred_proba_voting)

metrics_stacking, pred_stacking, pred_proba_stacking = evaluate_model(stacking_clf, X_test_scaled, y_test, 'Stacking Classifier')
results.append(metrics_stacking)
predictions['Stacking'] = (pred_stacking, pred_proba_stacking)

# Create results dataframe
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)
print(results_df.to_string(index=False))
print("="*80)

### 6.3 Confusion Matrices

In [ ]:
# Create confusion matrices for key models
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

model_names = ['XGBoost', 'LightGBM', 'CatBoost', 'LogisticRegression', 'Voting', 'Stacking']

for idx, (ax, model_name) in enumerate(zip(axes, model_names)):
    y_pred_model = predictions[model_name][0]
    cm = confusion_matrix(y_test, y_pred_model)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'{model_name}')
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    ax.set_xticklabels(['Non-Viral', 'Viral'])
    ax.set_yticklabels(['Non-Viral', 'Viral'])

plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Confusion matrices saved!")

### 6.4 Classification Reports

In [ ]:
# Print detailed classification reports for ensemble models
print("\n" + "="*80)
print("VOTING CLASSIFIER - Classification Report")
print("="*80)
print(classification_report(y_test, predictions['Voting'][0], 
                          target_names=['Non-Viral', 'Viral']))

print("\n" + "="*80)
print("STACKING CLASSIFIER - Classification Report")
print("="*80)
print(classification_report(y_test, predictions['Stacking'][0], 
                          target_names=['Non-Viral', 'Viral']))

## 7. Feature Importance Analysis

### 7.1 XGBoost Feature Importance

In [ ]:
# Get feature importance from XGBoost
xgb_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nXGBoost Top 15 Important Features:")
print(xgb_importance.head(15))

### 7.2 LightGBM Feature Importance

In [ ]:
# Get feature importance from LightGBM
lgb_importance = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nLightGBM Top 15 Important Features:")
print(lgb_importance.head(15))

### 7.3 Side-by-Side Feature Importance Comparison

In [ ]:
# Plot feature importances side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# XGBoost top 15
top_n = 15
xgb_top = xgb_importance.head(top_n).sort_values('importance')
axes[0].barh(range(len(xgb_top)), xgb_top['importance'].values, color='skyblue')
axes[0].set_yticks(range(len(xgb_top)))
axes[0].set_yticklabels(xgb_top['feature'].values)
axes[0].set_xlabel('Importance')
axes[0].set_title('XGBoost - Top 15 Feature Importances')
axes[0].grid(axis='x', alpha=0.3)

# LightGBM top 15
lgb_top = lgb_importance.head(top_n).sort_values('importance')
axes[1].barh(range(len(lgb_top)), lgb_top['importance'].values, color='lightcoral')
axes[1].set_yticks(range(len(lgb_top)))
axes[1].set_yticklabels(lgb_top['feature'].values)
axes[1].set_xlabel('Importance')
axes[1].set_title('LightGBM - Top 15 Feature Importances')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../models/feature_importances.png', dpi=300, bbox_inches='tight')
plt.show()
print("Feature importance plot saved!")

## 8. ROC Curves Comparison

In [ ]:
# Plot ROC curves for all models
fig, ax = plt.subplots(figsize=(12, 8))

models_to_plot = ['XGBoost', 'LightGBM', 'CatBoost', 'Voting', 'Stacking']
colors = plt.cm.tab10(np.linspace(0, 1, len(models_to_plot)))

for model_name, color in zip(models_to_plot, colors):
    y_pred_proba = predictions[model_name][1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC = {roc_auc:.3f})', color=color, linewidth=2)

# Plot diagonal
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Classifier')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../models/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()
print("ROC curves saved!")

## 9. Select and Save Best Model

### 9.1 Identify Best Model

In [ ]:
# Identify best model by F1-Macro score
best_model_idx = results_df['F1-Macro'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_f1_score = results_df.loc[best_model_idx, 'F1-Macro']

print(f"\nBest Model: {best_model_name}")
print(f"F1-Macro Score: {best_f1_score:.4f}")
print(f"\nBest Model Metrics:")
print(results_df.loc[best_model_idx].to_string())

### 9.2 Save Best Model

In [ ]:
# Select the best model object
best_model_mapping = {
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
    'CatBoost': catboost_model,
    'Logistic Regression': lr_model,
    'Voting Classifier': voting_clf,
    'Stacking Classifier': stacking_clf
}

best_model = best_model_mapping[best_model_name]

# Create models directory if it doesn't exist
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

# Save best model
model_save_path = models_dir / 'best_ensemble.pkl'
joblib.dump(best_model, model_save_path)

print(f"\nBest model saved to: {model_save_path}")
print(f"File size: {model_save_path.stat().st_size / 1024:.2f} KB")

### 9.3 Save Additional Model Artifacts

In [ ]:
# Save all models for reference
all_models = {
    'xgb': xgb_model,
    'lgb': lgb_model,
    'catboost': catboost_model,
    'voting': voting_clf,
    'stacking': stacking_clf
}

for name, model in all_models.items():
    model_path = models_dir / f'{name}_model.pkl'
    joblib.dump(model, model_path)
    print(f"Saved: {model_path}")

# Save scaler
scaler_path = models_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"Saved: {scaler_path}")

# Save label encoders
encoders_path = models_dir / 'label_encoders.pkl'
joblib.dump(label_encoders, encoders_path)
print(f"Saved: {encoders_path}")

# Save polynomial transformer for LR
poly_path = models_dir / 'polynomial_features.pkl'
joblib.dump(poly, poly_path)
print(f"Saved: {poly_path}")

# Save results dataframe
results_path = models_dir / 'evaluation_results.csv'
results_df.to_csv(results_path, index=False)
print(f"Saved: {results_path}")

## 10. Summary

In [ ]:
print("\n" + "="*80)
print("ENSEMBLE MODEL TRAINING - SUMMARY")
print("="*80)
print(f"\nDataset:")
print(f"  - Total samples: {len(df)}")
print(f"  - Training samples: {len(X_train)}")
print(f"  - Test samples: {len(X_test)}")
print(f"  - Total features: {X_train_scaled.shape[1]}")
print(f"  - Viral class distribution (train): {y_train.value_counts().to_dict()}")
print(f"  - Viral class distribution (test): {y_test.value_counts().to_dict()}")

print(f"\nModels Built:")
print(f"  1. XGBoost (base learner)")
print(f"  2. LightGBM (base learner, tuned)")
print(f"  3. CatBoost (base learner, with categorical features)")
print(f"  4. Logistic Regression (with polynomial features)")
print(f"  5. Voting Classifier (soft voting ensemble)")
print(f"  6. Stacking Classifier (meta-learner with passthrough)")

print(f"\nBest Model: {best_model_name}")
print(f"  - F1-Macro: {results_df.loc[best_model_idx, 'F1-Macro']:.4f}")
print(f"  - ROC-AUC: {results_df.loc[best_model_idx, 'ROC-AUC']:.4f}")
print(f"  - Accuracy: {results_df.loc[best_model_idx, 'Accuracy']:.4f}")
print(f"  - Precision: {results_df.loc[best_model_idx, 'Precision']:.4f}")
print(f"  - Recall: {results_df.loc[best_model_idx, 'Recall']:.4f}")

print(f"\nArtifacts Saved:")
print(f"  - best_ensemble.pkl (best model)")
print(f"  - All individual model pkl files")
print(f"  - scaler.pkl, label_encoders.pkl, polynomial_features.pkl")
print(f"  - evaluation_results.csv")
print(f"  - Feature importance and ROC curve plots")

print("\n" + "="*80)